# VerhulstPINN SoH — final result analysis

This notebook presents the frozen three-seed Step 11 results. It reads saved reports and figures only: it does **not** import the model, load a checkpoint, or train anything.

In [ ]:
import csv
import json
from pathlib import Path
from IPython.display import Image, Markdown, display

ROOT = Path.cwd()
if not (ROOT / 'artifacts').exists():
    ROOT = ROOT.parent
ANALYSIS = ROOT / 'artifacts' / 'verhulst' / 'analysis'
FIGURES = ANALYSIS / 'figures'
summary = json.loads((ANALYSIS / 'analysis_summary.json').read_text())

## Headline held-out test results

The ensemble is the equal-weight mean of predictions aligned by `(battery_id, cycle)`. Model selection used validation data only.

In [ ]:
m = summary['ensemble_metrics']
a = summary['across_seed_metrics']
table = f'''
| Result | MAE | RMSE | R² | Bias |
|---|---:|---:|---:|---:|
| Three-seed ensemble | {m['mae']:.6f} | {m['rmse']:.6f} | {m['r2']:.4f} | {m['bias']:.6f} |
| Seed mean ± SD | {a['mae']['mean']:.6f} ± {a['mae']['sample_std']:.6f} | {a['rmse']['mean']:.6f} ± {a['rmse']['sample_std']:.6f} | {a['r2']['mean']:.4f} ± {a['r2']['sample_std']:.4f} | {a['bias']['mean']:.6f} ± {a['bias']['sample_std']:.6f} |
'''
display(Markdown(table))

In [ ]:
display(Image(filename=str(FIGURES / '01_actual_vs_predicted_soh.png'), width=800))

## Representative degradation trajectories

Batteries are selected reproducibly as the lowest-, median-, and highest-RMSE test cases. The band shows ±1 standard deviation across the three fixed seeds.

In [ ]:
display(Image(filename=str(FIGURES / '02_representative_battery_trajectories.png'), width=1100))

## Error profile and seed stability

In [ ]:
for name in ('03_per_battery_errors', '04_error_distributions', '08_seed_stability'):
    display(Image(filename=str(FIGURES / f'{name}.png'), width=1050))

## Optimization and Verhulst physics

The physical state is `U = PCL = 1 − SoH`. The residual compares the autograd derivative of the surrogate with the analytical Verhulst right-hand side and is reported in PCL per physical cycle.

In [ ]:
for name in ('05_training_curves', '06_physics_residuals', '07_verhulst_rhs_vs_autograd'):
    display(Image(filename=str(FIGURES / f'{name}.png'), width=1050))

In [ ]:
p = summary['verhulst_parameters']
rows = []
for row in p['per_seed']:
    rows.append(f"| {row['seed']} | {row['rate_r_per_cycle']:.7f} | {row['capacity_K_pcl']:.5f} | {row['offset_C_pcl']:.5f} |")
display(Markdown('''
### Learned constrained parameters

| Seed | r (per cycle) | K (PCL) | C (PCL) |
|---:|---:|---:|---:|
''' + '\n'.join(rows)))

## Per-battery metrics

In [ ]:
with (ANALYSIS / 'per_battery_metrics.csv').open() as stream:
    batteries = list(csv.DictReader(stream))
preview = ['| Battery | Records | MAE | RMSE | R² | Bias |', '|---:|---:|---:|---:|---:|---:|']
for row in batteries:
    preview.append(f"| {row['battery_id']} | {row['records']} | {float(row['mae']):.5f} | {float(row['rmse']):.5f} | {float(row['r2']):.3f} | {float(row['bias']):.5f} |")
display(Markdown('\n'.join(preview)))

## Presentation guidance

Use the measured-versus-predicted plot and representative trajectories for the main Verhulst result slide. Use the residual and derivative-agreement figures to explain the physics constraint. Report that VerhulstPINN is the best SoH predictor in the current controlled comparison, while avoiding a statistical-significance claim from only three seeds.